## Setup
* nan_str에 '기타' 추가
* 사공인지_시기 처리 중 '-' 만 있을 시 빈 문자열이 생기는 경우 수정
* 증강 데이터 추가
* 인적사고, 물적사고 전처리 해제 (프롬프트에 쓰기 위함)
* 계절/기온 feature 추가

In [1]:
SEED = 42
import os
import numpy as np
import pandas as pd
import re
from datetime import datetime
from sklearn.model_selection import train_test_split as tts
from utils import *

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'  # Ubuntu
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [2]:
def convert_to_12hour_format(datetime_str):
    if pd.isna(datetime_str):
        return None
    datetime_str = datetime_str.strip()
    if datetime_str == "":
        return None
    # 시간 부분 추출
    date_part, meridiem, time_part  = datetime_str.split(' ')
    hours, minutes = time_part.split(':')
    # 00시를 12시로 변환
    if hours == '00':
        hours = '12'
    meridiem = "AM" if meridiem == "오전" else "PM"
    # 새로운 시간 문자열 생성
    new_datetime = f"{date_part} {hours}:{minutes} {meridiem}"
    return new_datetime

In [3]:
class CFG:
    debug = False
    dp_version = "a5"
    aug_version = "d1"
    target_var = "재발방지대책 및 향후조치계획"

In [4]:
createFolder(f"dataset/prepdata/{CFG.dp_version}")
seed_everything(SEED)

## Loading data

In [5]:
df_full = pd.read_csv('dataset/rawdata/train.csv')
df_test = pd.read_csv('dataset/rawdata/test.csv')
# augmented data
df_aug = pd.read_csv(f'dataset/augdata/{CFG.aug_version}/augdata.csv')
df_full[["aug_사고원인", f"aug_{CFG.target_var}"]] = df_aug[["사고원인", CFG.target_var]]

In [8]:
df_full["사고원인"].dropna().apply(len).describe()

count    23359.000000
mean        48.398262
std         38.748195
min          1.000000
25%         22.000000
50%         39.000000
75%         64.000000
max        738.000000
Name: 사고원인, dtype: float64

In [6]:
df_full["재발방지대책 및 향후조치계획"].apply(len).describe()

count    23422.000000
mean        60.187687
std         35.060058
min          7.000000
25%         36.000000
50%         51.000000
75%         75.000000
max        492.000000
Name: 재발방지대책 및 향후조치계획, dtype: float64

In [9]:
np.percentile(df_full["사고원인"].dropna().apply(len), [75, 95, 99])

array([ 64., 120., 184.])

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


In [8]:
np.percentile(df_full["재발방지대책 및 향후조치계획"].apply(len), [75, 95, 99])

array([ 75., 127., 185.])

In [7]:
df_full["aug_재발방지대책 및 향후조치계획"].apply(len).describe()

count    23422.000000
mean        68.710272
std         37.212004
min          8.000000
25%         45.000000
50%         61.000000
75%         84.000000
max        725.000000
Name: aug_재발방지대책 및 향후조치계획, dtype: float64

In [9]:
np.percentile(df_full["aug_재발방지대책 및 향후조치계획"].apply(len), [75, 95, 99])

array([ 84.  , 131.95, 195.  ])

In [6]:
df_full["발생일시"] = pd.to_datetime(df_full["발생일시"].apply(convert_to_12hour_format), format="%Y-%m-%d %I:%M %p")
df_test["발생일시"] = pd.to_datetime(df_test["발생일시"].apply(convert_to_12hour_format), format="%Y-%m-%d %I:%M %p")

In [7]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23422 entries, 0 to 23421
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   23422 non-null  object        
 1   발생일시                 23422 non-null  datetime64[ns]
 2   사고인지 시간              23422 non-null  object        
 3   날씨                   23422 non-null  object        
 4   기온                   23422 non-null  object        
 5   습도                   23422 non-null  object        
 6   공사종류                 23422 non-null  object        
 7   연면적                  23422 non-null  object        
 8   층 정보                 23422 non-null  object        
 9   인적사고                 23390 non-null  object        
 10  물적사고                 21932 non-null  object        
 11  공종                   23411 non-null  object        
 12  사고객체                 22735 non-null  object        
 13  작업프로세스               23359 non-

In [8]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 964 entries, 0 to 963
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ID       964 non-null    object        
 1   발생일시     964 non-null    datetime64[ns]
 2   사고인지 시간  964 non-null    object        
 3   날씨       964 non-null    object        
 4   기온       964 non-null    object        
 5   습도       964 non-null    object        
 6   공사종류     964 non-null    object        
 7   연면적      964 non-null    object        
 8   층 정보     964 non-null    object        
 9   인적사고     964 non-null    object        
 10  물적사고     964 non-null    object        
 11  공종       964 non-null    object        
 12  사고객체     963 non-null    object        
 13  작업프로세스   964 non-null    object        
 14  장소       964 non-null    object        
 15  부위       964 non-null    object        
 16  사고원인     964 non-null    object        
dtypes: datetime64[ns](1), object(16)
me

## Validation Split

In [9]:
df_tmp = df_full[df_full["발생일시"].dt.year == 2023].reset_index(drop=True)
df_train, df_valid = tts(df_tmp, test_size=1000, stratify=df_tmp["발생일시"].dt.month, random_state=SEED)
df_full = pd.concat([df_full[df_full["발생일시"].dt.year != 2023], df_train], axis=0).reset_index(drop=True)
df_valid = df_valid.reset_index(drop=True)

In [10]:
df_full.shape, df_valid.shape

((22422, 20), (1000, 20))

In [11]:
df_valid["발생일시"].dt.month.value_counts().sort_index()

발생일시
1     72
2     78
3     95
4     87
5     96
6     89
7     76
8     78
9     80
10    84
11    88
12    77
Name: count, dtype: int64

In [12]:
df_valid["발생일시"].dt.hour.value_counts().sort_index()

발생일시
0       3
2       1
4       1
5       4
6      10
7      47
8      99
9     118
10    158
11    100
12     27
13     87
14    112
15    121
16     73
17     21
18      9
19      4
20      1
22      2
23      2
Name: count, dtype: int64

## Preprocessing

In [13]:
class Preprocessor():
    def __init__(self):
        self.nan_str = ["", "-", "없음", "기타"]
        self.replace_str = "없음"
        self.var_info = {
            "dt": [],
            "str": [],
            "num": [],
            "cat": [],
        }
        self.num_imputer = {}

    @staticmethod
    def get_season(date: datetime) -> str:
        seasons = {
            (12, 1, 2): "겨울",
            (3, 4, 5): "봄",
            (6, 7, 8): "여름",
            (9, 10, 11): "가을"
        }
        for months, season in seasons.items():
            if date.month in months:
                return season
        return "모름"
            
    def text_preprocessing(self, x):
        text = "".join(re.sub(r"[^가-힣0-9a-z]", "", x.lower()).split())
        if text == "":
            return self.replace_str
        else:
            return text

    def feature_engineer(self, df, mode):
        for col in df.columns:
            if df[col].dtype.name == "object":
                df[col] = df[col].str.strip()
        # 발생일시
        df["발생일시_월"] = df["발생일시"].dt.month.fillna(-1).astype("str")
        df["발생일시_시간"] = df["발생일시"].dt.hour.fillna(-1).astype("str")
        # 날씨
        df["날씨"] = df["날씨"].fillna(self.replace_str).apply(self.text_preprocessing)
        # 사고인지 시간
        col = "사고인지 시간"
        tmp = df[col].str.split("-", n=1, expand=True)
        tmp.columns = ["사고인지_시기", "사고인지_일시"]
        tmp["사고인지_시기"] = tmp["사고인지_시기"].fillna(self.replace_str).apply(self.text_preprocessing)
        tmp["사고인지_일시"] = pd.to_datetime(tmp["사고인지_일시"].apply(convert_to_12hour_format), format="%Y-%m-%d %I:%M %p")
        tmp["사고인지_월"] = tmp["사고인지_일시"].dt.month.fillna(-1).astype("str")
        tmp["사고인지_시간"] = tmp["사고인지_일시"].dt.hour.fillna(-1).astype("str")
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 기온 & 습도
        df["기온"] = df["기온"].apply(lambda x: np.nan if x.strip().strip("℃") in self.nan_str else (x.strip().strip("℃"))).astype("float32")
        df["습도"] = df["습도"].apply(lambda x: np.nan if x.strip().strip("%") in self.nan_str else (x.strip().strip("%"))).astype("float32") 
        # 계절/기온
        df["계절"] = df["발생일시"].apply(self.get_season)
        df["계절/기온"] = df[["계절", "기온"]].apply(lambda x: f"{x['계절']}" + "/" + (f"{int(x['기온'])}°C" if pd.notna(x['기온']) else "모름"), axis=1)
        # 공사종류
        col = "공사종류"
        tmp = df[col].str.split("/", n=3, expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = [f"{col}_level{i}" for i in range(tmp.shape[1])]
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 연면적
        df["연면적"] = df["연면적"].apply(lambda x: np.nan if x.strip().strip("㎡").replace(",", "") in self.nan_str else (x.strip().strip("㎡").replace(",", ""))).astype("float32")
        # 층정보
        col = "층 정보"
        tmp = df[col].str.split(",", expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = ["지상층", "지하층"]
        tmp["지상층"] = tmp["지상층"].apply(lambda x: "-1.0" if x in self.nan_str else x.lstrip("지상").rstrip("층").strip()).astype("float32")
        tmp["지하층"] = tmp["지하층"].apply(lambda x: "-1.0" if x in self.nan_str else x.lstrip("지하").rstrip("층").strip()).astype("float32")
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 인적사고 & 물적사고
        df["인적사고"] = df["인적사고"].fillna(self.replace_str).apply(lambda x: " ".join(x.split()))
        df["물적사고"] = df["물적사고"].fillna(self.replace_str).apply(lambda x: " ".join(x.split()))
        # 공종
        col = "공종"
        tmp = df[col].str.split(">", n=1, expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = [f"{col}_level{i}" for i in range(tmp.shape[1])]
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 사고객체
        col = "사고객체"
        tmp = df[col].str.split(">", n=1, expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = [f"{col}_level{i}" for i in range(tmp.shape[1])]
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 작업프로세스
        col = "작업프로세스"
        tmp = df[col].str.split("및", n=1, expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = [f"{col}_level{i}" for i in range(tmp.shape[1])]
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 장소
        col = "장소"
        tmp = df[col].str.split("/", n=3, expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = [f"{col}_level{i}" for i in range(tmp.shape[1])]
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 부위
        col = "부위"
        tmp = df[col].str.split("/", n=3, expand=True).fillna(self.replace_str).apply(lambda x: x.apply(self.text_preprocessing))
        tmp.columns = [f"{col}_level{i}" for i in range(tmp.shape[1])]
        df = pd.concat([df.drop(col, axis=1), tmp], axis=1)
        # 사고원인
        df["사고원인"] = df["사고원인"].fillna(self.replace_str).apply(lambda x: " ".join(x.split()))
        if mode == "train":
            # nan imputation
            self.num_imputer["기온"] = df.groupby(df["발생일시_월"])["기온"].mean()
            self.num_imputer["습도"] = df.groupby(df["발생일시_월"])["습도"].mean()
            self.num_imputer["연면적"] = df["연면적"].mean()
            # data type processing
            self.var_info["dt"].extend(["발생일시", "사고인지_일시"])
            self.var_info["str"].append("사고원인")
            self.var_info["num"].extend(["기온", "습도", "연면적", "지상층", "지하층"])
            self.var_info["cat"] = diff(df.columns, [CFG.target_var] + self.var_info["dt"] + self.var_info["str"] + self.var_info["num"])
            assert sum([len(self.var_info[k]) for k in self.var_info]) + 1 == df.shape[1], "Variable information is not consistent"
        df.loc[df["기온"].isna(), "기온"] = df.loc[df["기온"].isna(), "발생일시_월"].map(self.num_imputer["기온"])
        df.loc[df["습도"].isna(), "습도"] = df.loc[df["습도"].isna(), "발생일시_월"].map(self.num_imputer["습도"])
        df.loc[df["연면적"].isna(), "연면적"] = self.num_imputer["연면적"]
        return df

    def run(self, df, mode="train"):
        # drop unnecessary feature
        df = df.drop("ID", axis=1)
        # drop duplicates & nan imputation
        if mode == "train":
            df = df.drop_duplicates().reset_index(drop=True)
            df["인적사고"] = df["인적사고"].fillna(self.replace_str)
            df["물적사고"] = df["물적사고"].fillna(self.replace_str)
            df = df.dropna().reset_index(drop=True)
        # feature engineering
        df = self.feature_engineer(df, mode)
        # drop duplicates
        if mode == "train":
            df = df.drop_duplicates().reset_index(drop=True)
            df[CFG.target_var] = df[CFG.target_var].apply(lambda x: " ".join(x.split()))
        return df

In [14]:
preprocessor = Preprocessor()

In [15]:
df_full = preprocessor.run(df_full.copy())
df_full.head(3)

,발생일시,날씨,기온,습도,연면적,인적사고,물적사고,사고원인,재발방지대책 및 향후조치계획,aug_사고원인,...,작업프로세스_level0,작업프로세스_level1,장소_level0,장소_level1,장소_level2,장소_level3,부위_level0,부위_level1,부위_level2,부위_level3
0,2022-12-31 07:20:00,맑음,0.0,5.0,6.634700e+04,넘어짐(미끄러짐),없음,밤새 결빙된 바닥에 미끄러짐,"제설작업 및 현장 내 결빙구간 제거, 근로자 아이젠 지급, 현장 근로자 안전사고 예...",얼어붙은 바닥에서 미끄러졌다,...,정리작업,없음,공동주택,외부,없음,없음,기타,앞,없음,없음
1,2022-12-31 02:20:00,맑음,-5.0,45.0,3.650641e+06,넘어짐(미끄러짐),기타,보호철판 덮개설치 후 빙판에서 미끄러져 넘어짐,"작업자 안전교육 실시, TBM 전 작업 종료 후 작업자 건강 상태 여부 확인, 일기...",보호철판 덮개 설치 이후 얼음길에서 미끄러져 넘어졌습니다,...,마감작업,없음,기타,외부,없음,없음,자재,바닥,없음,없음
2,2022-12-30 16:30:00,맑음,5.0,46.0,8.274200e+02,"절단, 베임",없음,원형 톱 사용 중 다른 인부와 부딪히면서 오른쪽 엄지손가락 손톱 부위 피부 손상,원형톱 사용 시 각별한 안전 준수 필요.,원형 톱 작업 중 다른 근로자와 충돌하여 오른쪽 엄지손가락 손톱 옆 피부가 다쳤습니다,...,절단작업,없음,근린생활시설,내부,없음,없음,공구류,앞,없음,없음


In [16]:
df_valid = preprocessor.run(df_valid.copy(), mode="test")
df_valid.head(3)

,발생일시,날씨,기온,습도,연면적,인적사고,물적사고,사고원인,재발방지대책 및 향후조치계획,aug_사고원인,...,사고객체_level0,사고객체_level1,작업프로세스_level0,작업프로세스_level1,장소_level0,장소_level1,장소_level2,부위_level0,부위_level1,부위_level2
0,2023-02-07 10:00:00,맑음,5.0,40.0,1306341.125,물체에 맞음,없음,"해체된 1,2단 거푸집을 정리하지않고 3단 거푸집을 해체하던 중 해체 거푸집이 선 ...","1, 2단 해체거푸집 정리 후 3단 거푸집 해체 실시.",3단 거푸집 해체 중 이미 해체된 1 2단 거푸집에 부딪혀 우측 발목 골절 부상을 ...,...,가시설,거푸집,해체작업,없음,공동주택,내부,없음,거푸집,고소,없음
1,2023-11-30 11:38:00,맑음,-1.0,42.0,3650641.250,질병,없음,STA. 39+320 철근콘크리트개거 자재정리 후 현장에서 점심식사 하러 가는중 쓰...,사고 발생 원인 파악 후 재발 방지 대책 마련 예정.,STA 39 320에서 철근콘크리트 개거 자재 정리를 마친 뒤 점심 식사를 하러 나...,...,질병,질병,기타,없음,도로,외부,없음,질병,바닥,없음
2,2023-01-25 12:40:00,맑음,-8.0,62.0,1456.000,넘어짐(물체에 걸림),없음,현장출입구에 깔아놓은 야자매트가 구겨져 밀려있어 갈고리로 현장내측으로 당기는 과정에...,"TBM 시 사고사례 전파, 관리감독자를 통한 근로자의 작업방법 및 위험요소 제거, ...",현장 출입구에 깔린 야자매트가 구겨져 밀려 있어 갈고리로 끌어당기다가 매트가 찢어지...,...,기타,기타,기타,없음,공동주택,외부,없음,기타,바닥,없음


In [17]:
df_test = preprocessor.run(df_test.copy(), mode="test")
df_test.head(3)

,발생일시,날씨,기온,습도,연면적,인적사고,물적사고,사고원인,발생일시_월,발생일시_시간,...,사고객체_level0,사고객체_level1,작업프로세스_level0,작업프로세스_level1,장소_level0,장소_level1,장소_level2,부위_level0,부위_level1,부위_level2
0,2024-06-03 09:39:00,맑음,27.0,53.0,1990.319946,부딪힘,전도,"펌프카 아웃트리거 바닥 고임목을 3단으로 보강 했음에도, 지반 침하(아웃트리거 우측...",6,9,...,건설기계,콘크리트펌프,타설작업,없음,교정및군사시설,외부,없음,콘크리트펌프,바닥,없음
1,2024-02-15 09:00:00,맑음,5.0,71.0,349895.000000,"절단, 베임",없음,작업자의 불안전한 행동(숫돌 측면 사용) 및 보안면 미 착용,2,9,...,건설공구,공구류,절단작업,없음,운수시설,내부,없음,공구류,핸드그라인더,없음
2,2024-02-01 09:30:00,흐림,5.0,58.0,171198.000000,떨어짐(2미터 미만),없음,작업자가 작업을 위해 이동 중 전방을 주시하지 않아 발을 헛디뎌 계단에서 굴러 넘어짐,2,9,...,기타,기타,이동,없음,공동주택,내부,없음,기타,바닥,없음


In [19]:
df_full.shape, df_valid.shape, df_test.shape

((21604, 39), (1000, 37), (964, 34))

## Save data

In [20]:
pickleIO(preprocessor.var_info, f"dataset/prepdata/{CFG.dp_version}/var_info.pkl", "w")
pickleIO(df_full, f"dataset/prepdata/{CFG.dp_version}/df_full.pkl", "w")
pickleIO(df_valid, f"dataset/prepdata/{CFG.dp_version}/df_valid.pkl", "w")
pickleIO(df_test, f"dataset/prepdata/{CFG.dp_version}/df_test.pkl", "w")